In [ ]:
import pandas as pd
import plotly.express as px

# Load data
df = pd.read_csv("../combined_occurrences_inaturalist-GBIF.csv")

/Users/mtwomack/My Drive (mitwomack@gmail.com)/Programming/Erdos Bootcamp 2026/Repos/summer26-suitability-of-bees/Data/Cleaning


In [ ]:

# Convert date column
df["eventDate"] = pd.to_datetime(df["eventDate"], errors="coerce")

# Western U.S. bounding box
lat_min = 25
lat_max = 49
lon_min = -125
lon_max = -93

# Filter to western U.S.
filtered_df = df[
    (df["decimalLatitude"] >= 31) &
    (df["decimalLatitude"] <= 49) &
    (df["decimalLongitude"] >= -125) &
    (df["decimalLongitude"] <= -93)
].copy()

# Remove locality = withheld
filtered_df = filtered_df[
    filtered_df["locality"].fillna("").str.lower() != "withheld"
].copy()

# Keep records from 1990 onward
filtered_df = filtered_df[
    filtered_df["eventDate"] >= "1980-01-01"
].copy()

# Remove same-date, same-location duplicates
filtered_df = filtered_df.drop_duplicates(
    subset=["eventDate", "decimalLatitude", "decimalLongitude"],
    keep="first"
).copy()

print(f"Original rows: {len(df)}")
print(f"Filtered rows: {len(filtered_df)}")
print(f"Rows removed: {len(df) - len(filtered_df)}")

# Save final cleaned file
filtered_df.to_csv("filtered_data.csv", index=False)

# Map the remaining records
fig = px.scatter_geo(
    filtered_df,
    lat="decimalLatitude",
    lon="decimalLongitude",
    scope="usa",
    projection="albers usa",
    hover_name="scientificName",
    hover_data={
        "eventDate": True,
        "locality": True,
        "decimalLatitude": True,
        "decimalLongitude": True
    },
    title="Occurrences in the Western United States"
)

fig.update_traces(
    marker=dict(
        size=5,
        opacity=0.6
    )
)

fig.show()